# Занятие 6, демо 1. Номер токена - метка, а не координата

Гауссово зашумление опирается на непрерывное представление, в котором есть
понятие малого сдвига: прибавили немного - получили близкую точку. Номера
токенов - целые числа, и сдвинуть их тоже можно. Посмотрим, чем это кончится.

## Одни и те же пары при двух нумерациях

Нумерацию токенов можно поменять, ничего не сломав, - если согласованно
переставить всё, что индексируется номером: строки эмбеддингов, строки выходного
слоя. Сами токены от этого не меняются.

In [ ]:
tokens = ["кот", "кошка", "собака", "стол", "бежать", "быстро"]
other = ["быстро", "стол", "кот", "бежать", "кошка", "собака"]

a = {t: i for i, t in enumerate(tokens)}
b = {t: i for i, t in enumerate(other)}

print(f"{'пара':>20} {'нумерация A':>12} {'нумерация B':>12}")
for x, y in (("кот", "кошка"), ("кот", "собака"), ("кот", "быстро")):
    print(f"{x + ' - ' + y:>20} {abs(a[x] - a[y]):>12} {abs(b[x] - b[y]):>12}")

## Не подобрана ли эта нумерация специально

Вторая нумерация выбрана руками, и на этом можно было бы поймать. Переберём
**все** нумерации этого словаря - их 720 - и посмотрим, насколько типичен
результат.

In [ ]:
from itertools import combinations, permutations

pairs = list(combinations(tokens, 2))
base = [abs(a[x] - a[y]) for x, y in pairs]

def changed(ids):
    return sum(abs(ids[x] - ids[y]) != d for (x, y), d in zip(pairs, base))

counts = [changed({t: i for i, t in enumerate(p)}) for p in permutations(tokens)]

print(f"всего нумераций:                       {len(counts)}")
print(f"сохраняют все {len(pairs)} расстояний:            {counts.count(0)}")
print(f"меняют хотя бы одно:                   {len(counts) - counts.count(0)}")
print(f"в среднем меняется расстояний:         {sum(counts) / len(counts):.2f}"
      f" из {len(pairs)}")
print(f"наша нумерация B меняет:               {changed(b)} из {len(pairs)}")

## От расстояний к самой порче

Расстояния - ещё половина дела. Тезис занятия про другое: про то, что **сама
операция порчи** зависит от произвольной нумерации.

Возьмём честное гауссово зашумление номера: $\mathrm{id}'=\mathrm{id}+\epsilon$,
$\epsilon\sim N(0,1)$, с округлением до ближайшего номера. Испортим один и тот же
токен при двух нумерациях и посмотрим, во что он превращается.

In [ ]:
import math

def corrupt(ids, token, sigma=1.0):
    """Распределение результата: id + N(0, sigma^2) с округлением."""
    normal = lambda z: 0.5 * (1 + math.erf(z / (sigma * math.sqrt(2))))
    centre = ids[token]
    out = {t: normal(i + 0.5 - centre) - normal(i - 0.5 - centre)
           for t, i in ids.items()}
    out["вне словаря"] = 1.0 - sum(out.values())
    return out

target = "стол"
pa, pb = corrupt(a, target), corrupt(b, target)

print(f"порча токена «{target}»: id + N(0,1), округление до ближайшего номера")
print()
print(f"{'во что превратился':>20} {'нумерация A':>12} {'нумерация B':>12}")
for t in tokens + ["вне словаря"]:
    print(f"{t:>20} {100 * pa[t]:>11.2f}% {100 * pb[t]:>11.2f}%")

## Что из этого следует

Одна и та же операция - «прибавить $N(0,1)$ к номеру» - при безобидной
перенумерации даёт другое распределение. «Стол» портится в «собаку» с
вероятностью 24% при одной нумерации и 0.02% при другой - разница в тысячу раз, и
взялась она из порядка строк в словаре. В «кота» - наоборот, 0.6% против 24%.

Значит $|\mathrm{id}(a)-\mathrm{id}(b)|$ - не инвариантная мера близости токенов,
а гауссов шум **непосредственно на номерах** не задаёт осмысленного понятия
«испортить чуть-чуть». Определить такой алгоритм можно, работать он будет; просто
то, что он делает, определяется произвольным выбором нумерации.

Важная оговорка, чтобы не увезти вывод дальше, чем он идёт: речь именно про
**сырые номера**. Непрерывная диффузия для текста существует - её строят в
пространстве эмбеддингов, в латентном пространстве автоэнкодера или на симплексе
распределений. Там непрерывная структура появляется не из номеров, а из
обученного представления.

Отсюда и второй путь: оставить дискретность и выбрать порчу, которой геометрия
не нужна вовсе, - замену токена на специальное состояние `[MASK]`. Маска не
«между» токенами, она означает «токен скрыт». Это не единственный вариант
дискретной порчи, но самый прямой.